# scRareBench | High-Level scDML | Dataset 3 — Wu breast-cancer atlas

> **Reproducibility**: Notebook sourced from the project execution archive, with outputs cleared and the scRareBench dependency pinned to the reviewed D3/D4 taxonomy commit `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`. Method configuration, method seed and dataset preprocessing settings are preserved.

This Colab notebook runs **scDML** on scRareBench Dataset 3
(`wu_breast_cancer_atlas`) through the generic:

`load_dataset(3) → MethodSpec(scDML) → benchmark_method() → HTML / multi-seed bundle`

Dataset-specific preparation remains package-owned. The notebook is pinned to
scRareBench commit `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`, where Dataset 3 itself is pinned to
the audited immutable CELLxGENE H5AD.

### Dataset 3 contract

- 100,064 cells × 28,468 features
- benchmark label: `celltype_subset` — **never supplied to scDML**
- integration/evaluation batch: `donor_id`
- canonical raw counts: `layers["counts"]`
- scIB-reference HVG policy: `global`
- registered rare-cell scenario metadata: package-owned

### scDML method configuration

To keep the method comparable with the previous scDML Dataset 0 workflow, this
notebook uses the same method-side settings:

- initial clustering: Leiden, resolution `3.0`
- scDML preprocessing: 2,000 HVGs, normalize-total `1e4`, log1p,
  batch-wise scaling with `max_value=10`, PCA 50
- fully unsupervised integration (`celltype_key=None`)
- `K_in=5`, `K_bw=10`, cosine neighbor metrics, `rule2`
- 50 training epochs, hard triplets, Euclidean metric, margin `0.2`
- output dimension 32, hidden layer `[256]`
- method seeds `42, 123, 2026`
- benchmark seed fixed at `42`

**Recommended Colab runtime:** GPU + High-RAM. scDML is legacy software and its
preprocessing/training path materializes dense matrices. For a first smoke run,
set `METHOD_SEEDS = [42]`.

## 1. Install the pinned scRareBench and scDML runtimes

In [ ]:
import subprocess
import sys

SCRAREBENCH_REVISION = "cfdb720f5488b9ceef9a225b35fc7558bcb0251c"
SCDML_REVISION = "d762dd75557c0dbb2d3039174d69a18195842b93"

SCRAREBENCH_GITHUB = (
    "git+https://github.com/amirhossein-alishahi/"
    f"scRareBench_.git@{SCRAREBENCH_REVISION}"
)
SCDML_GITHUB = (
    "git+https://github.com/eleozzr/"
    f"scDML.git@{SCDML_REVISION}"
)

# Bootstrap the exact scRareBench revision without allowing pip to alter
# the preinstalled scientific stack before runtime validation.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps",
    SCRAREBENCH_GITHUB,
])

# scDML imports both hnswlib and annoy at import time. These two revisions
# include modern Python support and are pinned for notebook reproducibility.
METHOD_RUNTIME_DEPENDENCIES = (
    "tables>=3.6.1",
    "numpy-groupies>=0.9.14",
    "numexpr>=2.7.1",
    "networkx>=2.5",
    "seaborn>=0.13",
    "torch>=2.0",
    "pytorch-metric-learning==2.9.0",
    "annoy @ git+https://github.com/spotify/annoy.git@379f744667aba6b40ba3db8a07678df173a88f74",
    "hnswlib @ git+https://github.com/nmslib/hnswlib.git@ca426729609b3221563047ceb269cc1213e0d376",
)

from scrarebench.runtime import setup_runtime, print_install_report

install_report = setup_runtime(
    extra_requirements=METHOD_RUNTIME_DEPENDENCIES,
    extra_imports=(
        "tables",
        "numpy_groupies",
        "numexpr",
        "networkx",
        "seaborn",
        "torch",
        "pytorch_metric_learning",
        "annoy",
        "hnswlib",
    ),
    quiet=False,
)
print_install_report(install_report)

# Install scDML itself without its historical dependency pins. The compatible
# runtime dependencies were installed/validated above.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps",
    SCDML_GITHUB,
])

# Fresh-process import smoke test catches missing/ABI-incompatible imports now,
# before downloading Dataset 3.
subprocess.check_call([
    sys.executable,
    "-c",
    (
        "import scDML, torch; "
        "from scDML import scDMLModel; "
        "print('scDML', getattr(scDML, '__version__', 'unknown')); "
        "print('torch', torch.__version__)"
    ),
])


## 2. Load Dataset 3 through scRareBench

In [ ]:
import scrarebench
from scrarebench import dataset_info, load_dataset

adata = load_dataset(3)
info = dataset_info(adata)

expected = {
    "dataset_key": "wu_breast_cancer_atlas",
    "label_key": "celltype_subset",
    "batch_key": "donor_id",
    "count_layer": "counts",
    "scib_hvg_batch_mode": "global",
    "benchmark_ready": True,
}
for key, value in expected.items():
    if info.get(key) != value:
        raise RuntimeError(
            f"Unexpected Dataset 3 contract for {key!r}: "
            f"observed={info.get(key)!r}, expected={value!r}"
        )

if adata.shape != (100_064, 28_468):
    raise RuntimeError(
        f"Unexpected Dataset 3 shape: {adata.shape}; "
        "expected (100064, 28468)."
    )
if "counts" not in adata.layers:
    raise RuntimeError("Dataset 3 canonical raw-count layer 'counts' is missing.")
if not adata.obs_names.is_unique:
    raise RuntimeError("Dataset 3 observation names are not unique.")

print("scRareBench:", scrarebench.__version__)
print(adata)
print("\nDataset contract:")
for key in (
    "dataset_key",
    "label_key",
    "batch_key",
    "count_layer",
    "scib_hvg_batch_mode",
    "benchmark_ready",
):
    print(f"  {key}: {info.get(key)}")


## 3. Reproducibility and method configuration

The scDML preprocessing below is **method-owned**. It is intentionally separate
from scRareBench's canonical scIB reference preprocessing.

- scDML: its native 2,000-HVG preprocessing
- scIB reference: package-controlled global 4,000-HVG policy for Dataset 3


In [ ]:
METHOD_SEEDS = [42, 123, 2026]   # use [42] for a first smoke run
BENCHMARK_SEED = 42

METHOD_CONFIG = {
    "cluster_method": "leiden",
    "resolution": 3.0,
    "n_high_var": 2000,
    "normalize_samples": True,
    "target_sum": 1e4,
    "log_normalize": True,
    "normalize_features": True,
    "pca_dim": 50,
    "scale_value": 10.0,

    "K_in": 5,
    "K_bw": 10,
    "K_in_metric": "cosine",
    "K_bw_metric": "cosine",
    "merge_rule": "rule2",

    "num_epochs": 50,
    "projection": False,
    "early_stop": False,
    "batch_size": 64,
    "metric": "euclidean",
    "margin": 0.2,
    "triplet_type": "hard",
    "device": None,  # scDML auto-selects CUDA when available

    "out_dim": 32,
    "emb_dim": [256],
    "save_model": False,
}

OUTPUT_DIR = "/content/scrarebench_scdml_dataset3_highlevel"
METHOD_WORK_DIR = "/content/scrarebench_scdml_dataset3_method_runs"


## 4. User-owned scDML adapter

scDML's legacy preprocessing concatenates batches and can change row order and
observation names. The guard column below preserves the original benchmark cell
identity, and the final latent is explicitly restored to the original
scRareBench order before it crosses the `MethodOutput` boundary.

No biological label is given to scDML.


In [ ]:
from pathlib import Path
import gc
import random
import shutil

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch

from scDML import scDMLModel
from scrarebench import MethodOutput, dataset_info

_ORIGINAL_BARCODE_COL = "_scrarebench_original_barcode"


def _light_raw_count_guard(matrix, *, seed, max_values=100_000):
    values = matrix.data if sp.issparse(matrix) else np.asarray(matrix).reshape(-1)
    values = np.asarray(values)

    if values.size > max_values:
        rng = np.random.default_rng(int(seed))
        values = values[
            rng.choice(values.size, size=int(max_values), replace=False)
        ]

    values = values.astype(float, copy=False)
    if values.size == 0:
        return

    if not np.isfinite(values).all():
        raise ValueError("Canonical count matrix contains NaN or infinite values.")
    if float(values.min()) < 0:
        raise ValueError("Canonical count matrix contains negative values.")
    if not np.allclose(values, np.rint(values), atol=1e-6, rtol=0.0):
        raise ValueError("Canonical count matrix does not look integer-like.")


def run_scdml(method_adata, seed, config):
    method_info = dataset_info(method_adata)
    batch_key = method_info["batch_key"]
    count_layer = method_info["count_layer"]

    if method_info.get("dataset_key") != "wu_breast_cancer_atlas":
        raise RuntimeError("This runner is configured for scRareBench Dataset 3.")
    if not count_layer or count_layer not in method_adata.layers:
        raise KeyError(
            f"Canonical raw-count layer {count_layer!r} is unavailable."
        )

    original_order = method_adata.obs_names.astype(str).to_numpy(copy=True)
    if len(np.unique(original_order)) != len(original_order):
        raise RuntimeError("Benchmark cell identifiers must be unique.")

    # Preserve identity across scDML's legacy AnnData.concatenate() preprocessing.
    method_adata.obs[_ORIGINAL_BARCODE_COL] = original_order

    counts = method_adata.layers[count_layer]
    _light_raw_count_guard(counts, seed=seed)

    # scDML.preprocess() consumes adata.X. This copy belongs only to this
    # per-seed method input; the benchmark AnnData remains untouched.
    method_adata.X = counts.copy()

    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

    seed_dir = Path(METHOD_WORK_DIR) / f"seed_{int(seed)}"
    if seed_dir.exists():
        shutil.rmtree(seed_dir)
    seed_dir.mkdir(parents=True, exist_ok=True)

    model = scDMLModel(
        verbose=True,
        save_dir=str(seed_dir) + "/",
    )

    adata_scdml = model.preprocess(
        method_adata,
        cluster_method=str(config["cluster_method"]),
        resolution=float(config["resolution"]),
        batch_key=batch_key,
        n_high_var=int(config["n_high_var"]),
        normalize_samples=bool(config["normalize_samples"]),
        target_sum=float(config["target_sum"]),
        log_normalize=bool(config["log_normalize"]),
        normalize_features=bool(config["normalize_features"]),
        pca_dim=int(config["pca_dim"]),
        scale_value=float(config["scale_value"]),
        mode="unsupervised",
    )

    if _ORIGINAL_BARCODE_COL not in adata_scdml.obs.columns:
        raise RuntimeError(
            "Original cell-identity guard was lost during scDML preprocessing."
        )
    if "init_cluster" not in adata_scdml.obs.columns:
        raise RuntimeError("scDML preprocessing did not create init_cluster.")

    num_init_clusters = int(adata_scdml.obs["init_cluster"].nunique())
    if num_init_clusters < 2:
        raise RuntimeError(
            f"scDML produced too few initial clusters: {num_init_clusters}"
        )

    # expect_num_cluster=None makes scDML estimate the target cluster count by
    # its eigengap heuristic. All candidate merge counts are therefore prepared.
    ncluster_candidates = list(range(2, num_init_clusters + 1))

    print(
        f"Seed {seed}: cells={adata_scdml.n_obs:,}, "
        f"features={adata_scdml.n_vars:,}, "
        f"batches={adata_scdml.obs[batch_key].nunique()}, "
        f"initial Leiden clusters={num_init_clusters}"
    )

    model.integrate(
        adata_scdml,
        batch_key=batch_key,
        ncluster_list=ncluster_candidates,
        expect_num_cluster=None,
        K_in=int(config["K_in"]),
        K_bw=int(config["K_bw"]),
        K_in_metric=str(config["K_in_metric"]),
        K_bw_metric=str(config["K_bw_metric"]),
        merge_rule=str(config["merge_rule"]),
        num_epochs=int(config["num_epochs"]),
        projection=bool(config["projection"]),
        early_stop=bool(config["early_stop"]),
        batch_size=int(config["batch_size"]),
        metric=str(config["metric"]),
        margin=float(config["margin"]),
        triplet_type=str(config["triplet_type"]),
        device=config["device"],
        seed=int(seed),
        out_dim=int(config["out_dim"]),
        emb_dim=list(config["emb_dim"]),
        save_model=bool(config["save_model"]),
        celltype_key=None,
        mode="unsupervised",
    )

    if "X_emb" not in adata_scdml.obsm:
        raise RuntimeError("scDML did not create adata.obsm['X_emb'].")

    latent_native = np.asarray(
        adata_scdml.obsm["X_emb"],
        dtype=np.float32,
    )
    native_barcodes = (
        adata_scdml.obs[_ORIGINAL_BARCODE_COL]
        .astype(str)
        .to_numpy()
    )

    if latent_native.ndim != 2:
        raise RuntimeError(
            f"scDML latent must be 2D; observed shape={latent_native.shape}."
        )
    if latent_native.shape[0] != len(native_barcodes):
        raise RuntimeError(
            "scDML latent rows do not match the post-preprocessing cells."
        )
    if not np.isfinite(latent_native).all():
        raise ValueError("scDML latent contains NaN or infinite values.")
    if len(np.unique(native_barcodes)) != len(native_barcodes):
        raise RuntimeError(
            "Cell identifiers are no longer unique after scDML preprocessing."
        )
    if set(native_barcodes) != set(original_order):
        raise RuntimeError(
            "scDML preprocessing changed the benchmark cell set."
        )

    # Restore the exact original scRareBench row order.
    position = pd.Series(
        np.arange(len(native_barcodes)),
        index=native_barcodes,
    )
    reorder = position.loc[original_order].to_numpy()
    latent = latent_native[reorder]

    if not np.array_equal(native_barcodes[reorder], original_order):
        raise RuntimeError(
            "Failed to restore scDML latent to the original benchmark order."
        )

    provenance = {}
    log_path = seed_dir / "log.txt"
    if log_path.exists() and log_path.stat().st_size > 0:
        provenance["scdml_log"] = log_path

    print(f"Seed {seed}: aligned scDML latent shape={latent.shape}")

    del model, adata_scdml, latent_native, position, reorder
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return MethodOutput(
        latent=latent,
        barcodes=original_order,
        representation_key="X_scdml",
        provenance_files=provenance,
    )


## 5. Run the high-level multi-seed benchmark

In [ ]:
from scrarebench import MethodSpec, benchmark_method

method = MethodSpec(
    name="scDML",
    runner=run_scdml,
    config=METHOD_CONFIG,
    dependencies=METHOD_RUNTIME_DEPENDENCIES + (SCDML_GITHUB,),
)

result = benchmark_method(
    adata,
    method,
    seeds=METHOD_SEEDS,
    output_dir=OUTPUT_DIR,
    benchmark_config={
        "random_state": BENCHMARK_SEED,
        "run_scib": True,
        "scib_n_hvg": 4000,
        "scib_reference_n_pcs": 50,
        "scib_n_jobs": 1,
        "scib_progress_bar": True,
        "scib_include_silhouette_batch": True,
        "scib_require_success": True,
        # Dataset 3's scIB HVG mode is read from package metadata (global).
        "write_pdf_report": False,
        "include_latent_in_bundle": False,
        "include_cell_ids": False,
    },
    install_dependencies=False,
    finalize=True,
    copy_input_per_seed=True,
)


## 6. Inspect the multi-seed result

In [ ]:
from IPython.display import display

display(result.summary().round(5))

print("Method:", result.method)
print("Completed seeds:", result.seeds)
print("Multi-seed HTML:", result.report_path)
print("Delivery archive:", result.archive_path)
print("Summary JSON:", result.summary_path)

for seed, run in result.runs.items():
    print(f"\nSeed {seed}")
    print("  representation:", run.representation_key)
    print("  output:", run.output_dir)
    print("  interactive report:", run.interactive_report_path)
    print("  bundle:", run.bundle_path)


## 7. Download the final multi-seed archive

In [ ]:
from pathlib import Path

archive_path = Path(result.archive_path)
if not archive_path.exists():
    raise FileNotFoundError(
        f"Multi-seed archive was not found: {archive_path}"
    )

print(f"Downloading: {archive_path.name}")
print(f"Size: {archive_path.stat().st_size / (1024**2):.2f} MB")

try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    pass
